# H1 2024 Dock Receiving Cutover Audit & Damaged Dock Loss Value

Audits the `2024-03-25` EDI 856 centi-unit cutover on `scm_manh_wmos.manh_wmos_inbound_receipts` and defines the **Damaged Dock Loss Value (`damaged_dock_loss_usd`)** financial metric across accepted vs. rejected dock receipts.


In [1]:
# Canonical SQL for Damaged Dock Loss Value (damaged_dock_loss_usd):
# Unlike OTIF (which filters to accepted receipts only), total physical dock damage loss
# values normalized damaged_eaches across accepted receipts using SAP PO Line unit_cost_usd
# bridged via manh_wmos_products (sku_id = skuId, itemNbr = item_nbr).
DAMAGED_DOCK_LOSS_SQL = '''
WITH deduped AS (
  SELECT
    receipt_id, po_number, item_nbr, dc_id, status, hauler_name,
    CASE WHEN DATE(received_at) < DATE '2024-03-25' THEN damaged_qty / 100.0 ELSE damaged_qty END AS damaged_eaches
  FROM `scm_manh_wmos.manh_wmos_inbound_receipts`
  QUALIFY ROW_NUMBER() OVER (PARTITION BY receipt_id ORDER BY received_at DESC) = 1
)
SELECT
  d.dc_id,
  ROUND(SUM(d.damaged_eaches * pol.unit_cost_usd), 2) AS damaged_dock_loss_usd
FROM deduped AS d
JOIN `scm_manh_wmos.manh_wmos_products` AS p ON d.item_nbr = p.itemNbr
JOIN `scm_sap_s4_mm.sap_s4_mm_po_lines` AS pol
  ON d.po_number = pol.po_number AND p.skuId = pol.sku_id
WHERE d.status IN ('received', 'putaway_completed')
GROUP BY d.dc_id
ORDER BY damaged_dock_loss_usd DESC
'''
print('Verified metric SQL: damaged_dock_loss_usd ready.')


Verified metric SQL: damaged_dock_loss_usd ready.
